In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

In [ ]:
import torch
import numpy as np
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler

from models.lstm_model import LSTM_Model
from utils import get_device

# Parameters

In [ ]:
ckpt_path = '../checkpoints/lstm_checkpoint.pt'
ticker    = 'VOO'
train_start = '2020-01-01'   # Must match training data start to refit scalers correctly
train_end   = '2026-04-25'   # Must match training data end

# Load Checkpoint

In [ ]:
device = get_device()

with open(ckpt_path, 'rb') as f:
    checkpoint = torch.load(f, map_location=device)

if not isinstance(checkpoint, dict):
    raise ValueError('Checkpoint must be a dict. Please retrain.')

required_keys = {'model_state_dict', 'model_config', 'preprocess_config'}
missing = required_keys - set(checkpoint.keys())
if missing:
    raise ValueError(f'Checkpoint missing keys: {sorted(missing)}')

## Extract config from checkpoint

In [ ]:
model_config      = checkpoint['model_config']
preprocess_config = checkpoint['preprocess_config']
target_cols       = checkpoint.get('target_cols', ['Close'])

seq_len    = preprocess_config.get('seq_length', 20)
input_cols = ['Open', 'High', 'Low', 'Close', 'Volume']

print(f'seq_len : {seq_len}')
print(f'model_config : {model_config}')
print(f'target_cols : {target_cols}')

## Build model and load weights

In [ ]:
model = LSTM_Model(**model_config).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
_ = model.eval()

print('Model loaded successfully.')

# Prepare Data

## Download training data to refit scalers
We refit `x_scaler` and `y_scaler` on the same training window used during training,
so the scale transformation is identical to what the model learned from.

In [ ]:
df = yf.download(ticker, start=train_start, end=train_end, auto_adjust=False)
df = df.dropna().copy()

x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

x_scaler.fit(df[input_cols].values)
y_scaler.fit(df[target_cols].values)

print(f'Fitted scalers on {len(df)} rows ({train_start} to {train_end})')

## Fetch latest data to build the input sequence

In [ ]:
df_latest = yf.download(ticker, period='6mo', interval='1d', auto_adjust=False)
df_latest = df_latest.dropna().copy()

if len(df_latest) < seq_len:
    raise ValueError(f'Not enough rows for seq_len={seq_len}. Got {len(df_latest)}.')

# Take the last seq_len rows as the input window
x_raw    = df_latest[input_cols].values[-seq_len:].astype(np.float32)
x_scaled = x_scaler.transform(x_raw)                       # (seq_len, n_features)

latest_date = df_latest.index[-1].date()
print(f'Using last {seq_len} rows. Latest data date: {latest_date}')

# Predict

In [ ]:
x_tensor = torch.tensor(x_scaled, dtype=torch.float32).unsqueeze(0).to(device)  # (1, seq_len, n_features)

with torch.no_grad():
    pred_scaled = model(x_tensor).cpu().numpy()   # (1, 1)

# Invert scale to get real price
pred_real = y_scaler.inverse_transform(pred_scaled)[0, 0]

print(f'Ticker          : {ticker}')
print(f'Latest date     : {latest_date}')
print(f'Predicted {target_cols[0]:5s} : {pred_real:.4f}')